In [1]:
import os
import io

import numpy as np

from typing import Tuple

import time
import cv2

from PIL import Image, ImageOps

import torch
import torchvision
import torch.onnx

from onnx_tf.backend import prepare
import onnx
from onnxsim import simplify

# import tvm
# import tvm.relay
# import tvm.contrib.graph_runtime as graph_runtime

from mobilenet_v2_tsm import MobileNetV2

import tensorflow as tf
from keras.models import load_model

import tensorflowjs as tfjs

# import warnings
# warnings.filterwarnings('ignore')

/Users/izakharkin/Desktop/skoltech/vrarhaptics/deepjest/phynder/convert/onnx-tensorflow/onnx_tf/common/__init__.py:96: UserWarning: onnx_tf.common.get_outputs_names is deprecated. It will be removed in future release. Use TensorflowGraph.get_outputs_names instead.
  warnings.warn(message)
Using TensorFlow backend.


In [2]:
SOFTMAX_THRES = 0
HISTORY_LOGIT = True
REFINE_OUTPUT = True

# def torch2tvm_module(torch_module: torch.nn.Module, torch_inputs: Tuple[torch.Tensor, ...], target):
#     torch_module.eval()
#     input_names = []
#     input_shapes = {}
#     with torch.no_grad():
#         for index, torch_input in enumerate(torch_inputs):
#             name = "i" + str(index)
#             input_names.append(name)
#             input_shapes[name] = torch_input.shape
#         buffer = io.BytesIO()
#         torch.onnx.export(torch_module, torch_inputs, buffer, input_names=input_names, output_names=["o" + str(i) for i in range(len(torch_inputs))])
#         outs = torch_module(*torch_inputs)
#         buffer.seek(0, 0)
#         onnx_model = onnx.load_model(buffer)
#         relay_module, params = tvm.relay.frontend.from_onnx(onnx_model, shape=input_shapes)
#     with tvm.relay.build_config(opt_level=3):
#         graph, tvm_module, params = tvm.relay.build(relay_module, target, params=params)
#     return graph, tvm_module, params


# def torch2executor(torch_module: torch.nn.Module, torch_inputs: Tuple[torch.Tensor, ...], target):
#     prefix = f"mobilenet_tsm_tvm_{target}"
#     lib_fname = f'{prefix}.tar'
#     graph_fname = f'{prefix}.json'
#     params_fname = f'{prefix}.params'
#     if os.path.exists(lib_fname) and os.path.exists(graph_fname) and os.path.exists(params_fname):
#         with open(graph_fname, 'rt') as f:
#             graph = f.read()
#         tvm_module = tvm.module.load(lib_fname)
#         params = tvm.relay.load_param_dict(bytearray(open(params_fname, 'rb').read()))
#     else:
#         graph, tvm_module, params = torch2tvm_module(torch_module, torch_inputs, target)
#         tvm_module.export_library(lib_fname)
#         with open(graph_fname, 'wt') as f:
#             f.write(graph)
#         with open(params_fname, 'wb') as f:
#             f.write(tvm.relay.save_param_dict(params))

#     ctx = tvm.gpu() if target.startswith('cuda') else tvm.cpu()
#     graph_module = graph_runtime.create(graph, tvm_module, ctx)
#     for pname, pvalue in params.items():
#         graph_module.set_input(pname, pvalue)

#     def executor(inputs: Tuple[tvm.nd.NDArray]):
#         for index, value in enumerate(inputs):
#             graph_module.set_input(index, value)
#         graph_module.run()
#         return tuple(graph_module.get_output(index) for index in range(len(inputs)))

#     return executor, ctx


# def get_executor(use_gpu=True):
#     torch_module = MobileNetV2(n_class=27)
#     if not os.path.exists("mobilenetv2_jester_online.pth.tar"):  # checkpoint not downloaded
#         print('Downloading PyTorch checkpoint...')
#         import urllib.request
#         url = 'https://file.lzhu.me/projects/tsm/models/mobilenetv2_jester_online.pth.tar'
#         urllib.request.urlretrieve(url, './mobilenetv2_jester_online.pth.tar')
#     torch_module.load_state_dict(torch.load("mobilenetv2_jester_online.pth.tar"))
#     torch_inputs = (torch.rand(1, 3, 224, 224),
#                     torch.zeros([1, 3, 56, 56]),
#                     torch.zeros([1, 4, 28, 28]),
#                     torch.zeros([1, 4, 28, 28]),
#                     torch.zeros([1, 8, 14, 14]),
#                     torch.zeros([1, 8, 14, 14]),
#                     torch.zeros([1, 8, 14, 14]),
#                     torch.zeros([1, 12, 14, 14]),
#                     torch.zeros([1, 12, 14, 14]),
#                     torch.zeros([1, 20, 7, 7]),
#                     torch.zeros([1, 20, 7, 7]))
#     if use_gpu:
#         target = 'cuda'
#     else:
#         target = 'llvm -mcpu=cortex-a72 -target=armv7l-linux-gnueabihf'
#     return torch2executor(torch_module, torch_inputs, target)


def transform(frame: np.ndarray):
    # 480, 640, 3, 0 ~ 255
    frame = cv2.resize(frame, (224, 224))  # (224, 224, 3) 0 ~ 255
    frame = frame / 255.0  # (224, 224, 3) 0 ~ 1.0
    frame = np.transpose(frame, axes=[2, 0, 1])  # (3, 224, 224) 0 ~ 1.0
    frame = np.expand_dims(frame, axis=0)  # (1, 3, 480, 640) 0 ~ 1.0
    return frame


class GroupScale(object):
    """ Rescales the input PIL.Image to the given 'size'.
    'size' will be the size of the smaller edge.
    For example, if height > width, then image will be
    rescaled to (size * height / width, size)
    size: size of the smaller edge
    interpolation: Default: PIL.Image.BILINEAR
    """

    def __init__(self, size, interpolation=Image.BILINEAR):
        self.worker = torchvision.transforms.Scale(size, interpolation)

    def __call__(self, img_group):
        return [self.worker(img) for img in img_group]


class GroupCenterCrop(object):
    def __init__(self, size):
        self.worker = torchvision.transforms.CenterCrop(size)

    def __call__(self, img_group):
        return [self.worker(img) for img in img_group]


class Stack(object):

    def __init__(self, roll=False):
        self.roll = roll

    def __call__(self, img_group):
        if img_group[0].mode == 'L':
            return np.concatenate([np.expand_dims(x, 2) for x in img_group], axis=2)
        elif img_group[0].mode == 'RGB':
            if self.roll:
                return np.concatenate([np.array(x)[:, :, ::-1] for x in img_group], axis=2)
            else:
                return np.concatenate(img_group, axis=2)


class ToTorchFormatTensor(object):
    """ Converts a PIL.Image (RGB) or numpy.ndarray (H x W x C) in the range [0, 255]
    to a torch.FloatTensor of shape (C x H x W) in the range [0.0, 1.0] """

    def __init__(self, div=True):
        self.div = div

    def __call__(self, pic):
        if isinstance(pic, np.ndarray):
            # handle numpy array
            img = torch.from_numpy(pic).permute(2, 0, 1).contiguous()
        else:
            # handle PIL Image
            img = torch.ByteTensor(torch.ByteStorage.from_buffer(pic.tobytes()))
            img = img.view(pic.size[1], pic.size[0], len(pic.mode))
            # put it from HWC to CHW format
            # yikes, this transpose takes 80% of the loading time/CPU
            img = img.transpose(0, 1).transpose(0, 2).contiguous()
        return img.float().div(255) if self.div else img.float()


class GroupNormalize(object):
    def __init__(self, mean, std):
        self.mean = mean
        self.std = std

    def __call__(self, tensor):
        rep_mean = self.mean * (tensor.size()[0] // len(self.mean))
        rep_std = self.std * (tensor.size()[0] // len(self.std))

        # TODO: make efficient
        for t, m, s in zip(tensor, rep_mean, rep_std):
            t.sub_(m).div_(s)

        return tensor


def get_transform():
    cropping = torchvision.transforms.Compose([
        GroupScale(256),
        GroupCenterCrop(224),
    ])
    transform = torchvision.transforms.Compose([
        cropping,
        Stack(roll=False),
        ToTorchFormatTensor(div=True),
        GroupNormalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])
    return transform

catigories = [
    "Doing other things",  # 0
    "Drumming Fingers",  # 1
    "No gesture",  # 2
    "Pulling Hand In",  # 3
    "Pulling Two Fingers In",  # 4
    "Pushing Hand Away",  # 5
    "Pushing Two Fingers Away",  # 6
    "Rolling Hand Backward",  # 7
    "Rolling Hand Forward",  # 8
    "Shaking Hand",  # 9
    "Sliding Two Fingers Down",  # 10
    "Sliding Two Fingers Left",  # 11
    "Sliding Two Fingers Right",  # 12
    "Sliding Two Fingers Up",  # 13
    "Stop Sign",  # 14
    "Swiping Down",  # 15
    "Swiping Left",  # 16
    "Swiping Right",  # 17
    "Swiping Up",  # 18
    "Thumb Down",  # 19
    "Thumb Up",  # 20
    "Turning Hand Clockwise",  # 21
    "Turning Hand Counterclockwise",  # 22
    "Zooming In With Full Hand",  # 23
    "Zooming In With Two Fingers",  # 24
    "Zooming Out With Full Hand",  # 25
    "Zooming Out With Two Fingers"  # 26
]


n_still_frame = 0

def process_output(idx_, history):
    # idx_: the output of current frame
    # history: a list containing the history of predictions
    if not REFINE_OUTPUT:
        return idx_, history

    max_hist_len = 20  # max history buffer

    # mask out illegal action
    if idx_ in [7, 8, 21, 22, 3]:
        idx_ = history[-1]

    # use only single no action class
    if idx_ == 0:
        idx_ = 2
    
    # history smoothing
    if idx_ != history[-1]:
        if not (history[-1] == history[-2]): #  and history[-2] == history[-3]):
            idx_ = history[-1]
    

    history.append(idx_)
    history = history[-max_hist_len:]

    return history[-1], history

In [3]:
def torch2onnx(
    torch_module: torch.nn.Module, 
    torch_inputs: Tuple[torch.Tensor, ...], 
    onnx_path
):
    torch_module.eval()
    input_names = []
    input_shapes = {}
    with torch.no_grad():
        for index, torch_input in enumerate(torch_inputs):
            name = "i" + str(index)
            input_names.append(name)
            input_shapes[name] = torch_input.shape
        with open(onnx_path, 'wb') as model_file:
            torch.onnx.export(
                torch_module, 
                torch_inputs, 
                model_file, 
                input_names=input_names, 
                output_names=["o" + str(i) for i in range(len(torch_inputs))],
                opset_version=10
            )

* Load the model:

In [21]:
os.makedirs('./models', exist_ok=True)
TORCH_MODEL_PATH= './models/mobilenetv2_jester_online.pth.tar'
torch_module = MobileNetV2(n_class=27)
if not os.path.exists(TORCH_MODEL_PATH):  # checkpoint not downloaded
    print('Downloading PyTorch checkpoint...')
    import urllib.request
    url = 'https://file.lzhu.me/projects/tsm/models/mobilenetv2_jester_online.pth.tar'
    urllib.request.urlretrieve(url, TORCH_MODEL_PATH)
torch_module.load_state_dict(torch.load(TORCH_MODEL_PATH))
torch_module.eval()

MobileNetV2(
  (features): ModuleList(
    (0): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
        (3): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (4): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
       

* Convert to ONNX:

In [5]:
ONNX_MODEL_PATH = './models/jestnet.onnx'

torch_inputs = (torch.rand(1, 3, 224, 224, dtype=torch.float32),
                torch.zeros([1, 3, 56, 56], dtype=torch.float32),
                torch.zeros([1, 4, 28, 28], dtype=torch.float32),
                torch.zeros([1, 4, 28, 28], dtype=torch.float32),
                torch.zeros([1, 8, 14, 14], dtype=torch.float32),
                torch.zeros([1, 8, 14, 14], dtype=torch.float32),
                torch.zeros([1, 8, 14, 14], dtype=torch.float32),
                torch.zeros([1, 12, 14, 14], dtype=torch.float32),
                torch.zeros([1, 12, 14, 14], dtype=torch.float32),
                torch.zeros([1, 20, 7, 7], dtype=torch.float32),
                torch.zeros([1, 20, 7, 7], dtype=torch.float32))

for param in torch_module.parameters():
    param = param.float()

for module in torch_module.children():
    for module_1 in module.children():
        for module_2 in module_1.children():
            for module_3 in module_2.children():
                if hasattr(module_3, 'num_batches_tracked'):
                    module_3.num_batches_tracked = module_3.num_batches_tracked.float()
#                 if str(module_3).split('(')[0] == 'BatchNorm2d':
#                     print(module_3)

torch2onnx(
    torch_module=torch_module, 
    torch_inputs=torch_inputs, 
    onnx_path=ONNX_MODEL_PATH
)

/Users/izakharkin/Desktop/skoltech/vrarhaptics/deepjest/phynder/convert/mobilenet_v2_tsm.py:95: TracerWarning: Converting a tensor to a Python index might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  x1, x2 = x[:, : c // 8], x[:, c // 8:]


* Load from ONNX:

In [6]:
with open(ONNX_MODEL_PATH, 'rb') as onnx_model_file:
    onnx_model = onnx.load_model(onnx_model_file)
onnx.checker.check_model(onnx_model)

In [7]:
print(onnx.helper.printable_graph(onnx_model.graph))

graph torch-jit-export (
  %i0[FLOAT, 1x3x224x224]
  %i1[FLOAT, 1x3x56x56]
  %i2[FLOAT, 1x4x28x28]
  %i3[FLOAT, 1x4x28x28]
  %i4[FLOAT, 1x8x14x14]
  %i5[FLOAT, 1x8x14x14]
  %i6[FLOAT, 1x8x14x14]
  %i7[FLOAT, 1x12x14x14]
  %i8[FLOAT, 1x12x14x14]
  %i9[FLOAT, 1x20x7x7]
  %i10[FLOAT, 1x20x7x7]
) initializers (
  %classifier.bias[FLOAT, 27]
  %classifier.weight[FLOAT, 27x1280]
  %features.0.0.weight[FLOAT, 32x3x3x3]
  %features.0.1.bias[FLOAT, 32]
  %features.0.1.num_batches_tracked[INT64, scalar]
  %features.0.1.running_mean[FLOAT, 32]
  %features.0.1.running_var[FLOAT, 32]
  %features.0.1.weight[FLOAT, 32]
  %features.1.conv.0.weight[FLOAT, 32x1x3x3]
  %features.1.conv.1.bias[FLOAT, 32]
  %features.1.conv.1.num_batches_tracked[FLOAT, scalar]
  %features.1.conv.1.running_mean[FLOAT, 32]
  %features.1.conv.1.running_var[FLOAT, 32]
  %features.1.conv.1.weight[FLOAT, 32]
  %features.1.conv.3.weight[FLOAT, 16x32x1x1]
  %features.1.conv.4.bias[FLOAT, 16]
  %features.1.conv.4.num_batches_tracke

* [optional] Simplify:

In [8]:
model_simp, check = simplify(onnx_model)
assert check, "Simplified ONNX model could not be validated"

In [9]:
print('Before', onnx_model.ByteSize())
print('After', model_simp.ByteSize())

Before 9211341
After 8981232


In [10]:
print(onnx.helper.printable_graph(model_simp.graph))

graph torch-jit-export (
  %i0[FLOAT, 1x3x224x224]
  %i1[FLOAT, 1x3x56x56]
  %i2[FLOAT, 1x4x28x28]
  %i3[FLOAT, 1x4x28x28]
  %i4[FLOAT, 1x8x14x14]
  %i5[FLOAT, 1x8x14x14]
  %i6[FLOAT, 1x8x14x14]
  %i7[FLOAT, 1x12x14x14]
  %i8[FLOAT, 1x12x14x14]
  %i9[FLOAT, 1x20x7x7]
  %i10[FLOAT, 1x20x7x7]
) initializers (
  %classifier.bias[FLOAT, 27]
  %classifier.weight[FLOAT, 27x1280]
  %351[INT64, 1]
  %360[INT64, 1]
  %390[INT64, 1]
  %399[INT64, 1]
  %421[INT64, 1]
  %430[INT64, 1]
  %460[INT64, 1]
  %469[INT64, 1]
  %491[INT64, 1]
  %500[INT64, 1]
  %522[INT64, 1]
  %531[INT64, 1]
  %561[INT64, 1]
  %570[INT64, 1]
  %592[INT64, 1]
  %601[INT64, 1]
  %631[INT64, 1]
  %640[INT64, 1]
  %662[INT64, 1]
  %671[INT64, 1]
  %788[FLOAT, 32x3x3x3]
  %790[FLOAT, 32]
  %792[FLOAT, 32x1x3x3]
  %794[FLOAT, 32]
  %796[FLOAT, 16x32x1x1]
  %798[FLOAT, 16]
  %800[FLOAT, 96x16x1x1]
  %802[FLOAT, 96]
  %804[FLOAT, 96x1x3x3]
  %806[FLOAT, 96]
  %808[FLOAT, 24x96x1x1]
  %810[FLOAT, 24]
  %812[FLOAT, 144x24x1x1]
  %

In [11]:
ONNX_SIMPLE_MODEL_PATH = './models/jestnet_simple.onnx'
onnx.save(model_simp, ONNX_SIMPLE_MODEL_PATH)

* ONNX runtime check:

In [12]:
import onnxruntime
ort_session = onnxruntime.InferenceSession(ONNX_SIMPLE_MODEL_PATH)

In [13]:
input_names = [ort_session.get_inputs()[i].name for i in range(len(ort_session.get_inputs()))]
input_names

['i0', 'i1', 'i2', 'i3', 'i4', 'i5', 'i6', 'i7', 'i8', 'i9', 'i10']

In [14]:
output_names = [ort_session.get_outputs()[i].name for i in range(len(ort_session.get_inputs()))]
output_names

['o0', 'o1', 'o2', 'o3', 'o4', 'o5', 'o6', 'o7', 'o8', 'o9', 'o10']

In [15]:
np_inputs = [
    np.random.rand(1, 3, 224, 224),
    np.random.rand(1, 3, 56, 56),
    np.random.rand(1, 4, 28, 28),
    np.random.rand(1, 4, 28, 28),
    np.random.rand(1, 8, 14, 14),
    np.random.rand(1, 8, 14, 14),
    np.random.rand(1, 8, 14, 14),
    np.random.rand(1, 12, 14, 14),
    np.random.rand(1, 12, 14, 14),
    np.random.rand(1, 20, 7, 7),
    np.random.rand(1, 20, 7, 7)
]
np_inputs = [np_input.astype(np.float32) for np_input in np_inputs]

In [16]:
%%time
outputs = ort_session.run(output_names, {input_names[i]: np_inputs[i] for i in range(len(np_inputs))})

CPU times: user 29.3 ms, sys: 17.7 ms, total: 47 ms
Wall time: 11.5 ms


In [17]:
for i in range(len(output_names)):
    print(outputs[i].shape)

(1, 27)
(1, 3, 56, 56)
(1, 4, 28, 28)
(1, 4, 28, 28)
(1, 8, 14, 14)
(1, 8, 14, 14)
(1, 8, 14, 14)
(1, 12, 14, 14)
(1, 12, 14, 14)
(1, 20, 7, 7)
(1, 20, 7, 7)


* tflite2onnx:

In [4]:
import tflite2onnx

tflite_path = '/Users/izakharkin/Desktop/inclusio/Inclusio/mediapipe/mediapipe/models/hand_landmark.tflite'
onnx_path = './models/hand_landmark.onnx'

tflite2onnx.convert(tflite_path, onnx_path)

NotImplementedError: Unsupported TFLite OP: 6

* ONNX -> TensorFlow:

In [18]:
tf_rep = prepare(onnx_model)

2020-06-28 23:37:39,458 - onnx-tf - INFO - Fail to get since_version of BitShift in domain `` with max_inclusive_version=10. Set to 1.
2020-06-28 23:37:39,460 - onnx-tf - INFO - Unknown op ConstantFill in domain `ai.onnx`.
2020-06-28 23:37:39,460 - onnx-tf - INFO - Fail to get since_version of CumSum in domain `` with max_inclusive_version=10. Set to 1.
2020-06-28 23:37:39,461 - onnx-tf - INFO - Fail to get since_version of Det in domain `` with max_inclusive_version=10. Set to 1.
2020-06-28 23:37:39,462 - onnx-tf - INFO - Fail to get since_version of DynamicQuantizeLinear in domain `` with max_inclusive_version=10. Set to 1.
2020-06-28 23:37:39,464 - onnx-tf - INFO - Fail to get since_version of GatherND in domain `` with max_inclusive_version=10. Set to 1.
2020-06-28 23:37:39,465 - onnx-tf - INFO - Unknown op ImageScaler in domain `ai.onnx`.
2020-06-28 23:37:39,467 - onnx-tf - INFO - Fail to get since_version of Range in domain `` with max_inclusive_version=10. Set to 1.
2020-06-28 2

Instructions for updating:
Create a `tf.sparse.SparseTensor` and use `tf.sparse.to_dense` instead.


In [19]:
print(tf_rep.inputs) # Input nodes to the model
print('-----')
print(tf_rep.outputs) # Output nodes from the model
print('-----')
print(tf_rep.tensor_dict) # All nodes in the model

['i0', 'i1', 'i2', 'i3', 'i4', 'i5', 'i6', 'i7', 'i8', 'i9', 'i10']
-----
['o0', 'o1', 'o2', 'o3', 'o4', 'o5', 'o6', 'o7', 'o8', 'o9', 'o10']
-----
{'classifier.bias': <tf.Tensor 'classifier.bias:0' shape=(27,) dtype=float32>, 'classifier.weight': <tf.Tensor 'classifier.weight:0' shape=(27, 1280) dtype=float32>, 'features.0.0.weight': <tf.Tensor 'features.0.0.weight:0' shape=(32, 3, 3, 3) dtype=float32>, 'features.0.1.bias': <tf.Tensor 'features.0.1.bias:0' shape=(32,) dtype=float32>, 'features.0.1.num_batches_tracked': <tf.Tensor 'features.0.1.num_batches_tracked:0' shape=() dtype=int64>, 'features.0.1.running_mean': <tf.Tensor 'features.0.1.running_mean:0' shape=(32,) dtype=float32>, 'features.0.1.running_var': <tf.Tensor 'features.0.1.running_var:0' shape=(32,) dtype=float32>, 'features.0.1.weight': <tf.Tensor 'features.0.1.weight:0' shape=(32,) dtype=float32>, 'features.1.conv.0.weight': <tf.Tensor 'features.1.conv.0.weight:0' shape=(32, 1, 3, 3) dtype=float32>, 'features.1.conv.1.

In [20]:
TF_MODEL_PATH = './models/jestnet_tf.pb'
tf_rep.export_graph(TF_MODEL_PATH)

* Simplified -> TensorFlow:

In [21]:
tf_rep = prepare(model_simp, strict=False)

2020-06-28 23:37:46,517 - onnx-tf - INFO - Fail to get since_version of BitShift in domain `` with max_inclusive_version=10. Set to 1.
2020-06-28 23:37:46,518 - onnx-tf - INFO - Unknown op ConstantFill in domain `ai.onnx`.
2020-06-28 23:37:46,519 - onnx-tf - INFO - Fail to get since_version of CumSum in domain `` with max_inclusive_version=10. Set to 1.
2020-06-28 23:37:46,520 - onnx-tf - INFO - Fail to get since_version of Det in domain `` with max_inclusive_version=10. Set to 1.
2020-06-28 23:37:46,521 - onnx-tf - INFO - Fail to get since_version of DynamicQuantizeLinear in domain `` with max_inclusive_version=10. Set to 1.
2020-06-28 23:37:46,522 - onnx-tf - INFO - Fail to get since_version of GatherND in domain `` with max_inclusive_version=10. Set to 1.
2020-06-28 23:37:46,523 - onnx-tf - INFO - Unknown op ImageScaler in domain `ai.onnx`.
2020-06-28 23:37:46,524 - onnx-tf - INFO - Fail to get since_version of Range in domain `` with max_inclusive_version=10. Set to 1.
2020-06-28 2

In [22]:
print(tf_rep.inputs) # Input nodes to the model
print('-----')
print(tf_rep.outputs) # Output nodes from the model
print('-----')
print(tf_rep.tensor_dict) # All nodes in the model

['i0', 'i1', 'i2', 'i3', 'i4', 'i5', 'i6', 'i7', 'i8', 'i9', 'i10']
-----
['o0', 'o1', 'o2', 'o3', 'o4', 'o5', 'o6', 'o7', 'o8', 'o9', 'o10']
-----
{'classifier.bias': <tf.Tensor 'classifier.bias:0' shape=(27,) dtype=float32>, 'classifier.weight': <tf.Tensor 'classifier.weight:0' shape=(27, 1280) dtype=float32>, '351': <tf.Tensor '351:0' shape=(1,) dtype=int64>, '360': <tf.Tensor '360:0' shape=(1,) dtype=int64>, '390': <tf.Tensor '390:0' shape=(1,) dtype=int64>, '399': <tf.Tensor '399:0' shape=(1,) dtype=int64>, '421': <tf.Tensor '421:0' shape=(1,) dtype=int64>, '430': <tf.Tensor '430:0' shape=(1,) dtype=int64>, '460': <tf.Tensor '460:0' shape=(1,) dtype=int64>, '469': <tf.Tensor '469:0' shape=(1,) dtype=int64>, '491': <tf.Tensor '491:0' shape=(1,) dtype=int64>, '500': <tf.Tensor '500:0' shape=(1,) dtype=int64>, '522': <tf.Tensor '522:0' shape=(1,) dtype=int64>, '531': <tf.Tensor '531:0' shape=(1,) dtype=int64>, '561': <tf.Tensor '561:0' shape=(1,) dtype=int64>, '570': <tf.Tensor '570:

In [23]:
TF_SIMPLE_MODEL_PATH = './models/jestnet_simple_tf.pb'
tf_rep.export_graph(TF_SIMPLE_MODEL_PATH)

* TensorFlow -> tf.js:

```
tensorflowjs_converter --input_format=tf_frozen_model --output_node_names='o0,o1,o2,o3,o4,o5,o6,o7,o8,o9,o10' ./jestnet_tf.pb ./jestnet
```

```
tensorflowjs_converter --input_format=tf_frozen_model --output_node_names='o0,o1,o2,o3,o4,o5,o6,o7,o8,o9,o10' ./jestnet_simple_tf.pb ./jestnet_simple
```

In [24]:
import time

import numpy as np

import tensorflow as tf
print(tf.__version__)

tf.compat.v1.disable_eager_execution()

2.2.0


[Very useful link](https://blog.metaflow.fr/tensorflow-how-to-freeze-a-model-and-serve-it-with-a-python-api-d4f3596b3adc)

In [25]:
def load_graph(frozen_graph_filename):
    # We load the protobuf file from the disk and parse it to retrieve the 
    # unserialized graph_def
    with tf.io.gfile.GFile(frozen_graph_filename, "rb") as f:
        graph_def = tf.compat.v1.GraphDef()
        graph_def.ParseFromString(f.read())

    # Then, we import the graph_def into a new Graph and returns it 
    with tf.Graph().as_default() as graph:
        # The name var will prefix every op/nodes in your graph
        # Since we load everything in a new graph, this is not needed
        tf.import_graph_def(graph_def, name="prefix")
    return graph

In [26]:
TF_SIMPLE_MODEL_PATH = './models/jestnet_simple_tf.pb'
graph = load_graph(TF_SIMPLE_MODEL_PATH)

In [27]:
dir(graph)

['_ControlDependenciesController',
 '__class__',
 '__delattr__',
 '__dict__',
 '__dir__',
 '__doc__',
 '__eq__',
 '__format__',
 '__ge__',
 '__getattribute__',
 '__gt__',
 '__hash__',
 '__init__',
 '__init_subclass__',
 '__le__',
 '__lt__',
 '__module__',
 '__ne__',
 '__new__',
 '__reduce__',
 '__reduce_ex__',
 '__repr__',
 '__setattr__',
 '__sizeof__',
 '__str__',
 '__subclasshook__',
 '__weakref__',
 '_add_control_dependencies',
 '_add_device_to_stack',
 '_add_function',
 '_add_new_tf_operations',
 '_add_op',
 '_apply_device_functions',
 '_as_graph_def',
 '_as_graph_element_locked',
 '_attr_scope',
 '_attr_scope_map',
 '_auto_cast_variable_read_dtype',
 '_bcast_grad_args_cache',
 '_building_function',
 '_c_graph',
 '_check_not_finalized',
 '_collections',
 '_colocate_with_for_gradient',
 '_colocation_stack',
 '_container',
 '_control_dependencies_for_inputs',
 '_control_dependencies_stack',
 '_control_flow_context',
 '_copy_functions_to_graph_def',
 '_create_op_from_tf_operation',
 '

In [28]:
# graph.get_operations()

In [29]:
# i0 = tf.compat.v1.placeholder(tf.float32, shape=[None, 3, 224, 224], name="i0")
# i1 = tf.compat.v1.placeholder(tf.float32, shape=[None, 3, 56, 56], name="i1")
# i2 = tf.compat.v1.placeholder(tf.float32, shape=[None, 4, 28, 28], name="i2")
# i3 = tf.compat.v1.placeholder(tf.float32, shape=[None, 4, 28, 28], name="i3")
# i4 = tf.compat.v1.placeholder(tf.float32, shape=[None, 8, 14, 14], name="i4")
# i5 = tf.compat.v1.placeholder(tf.float32, shape=[None, 8, 14, 14], name="i5")
# i6 = tf.compat.v1.placeholder(tf.float32, shape=[None, 8, 14, 14], name="i6")
# i7 = tf.compat.v1.placeholder(tf.float32, shape=[None, 12, 14, 14], name="i7")
# i8 = tf.compat.v1.placeholder(tf.float32, shape=[None, 12, 14, 14], name="i8")
# i9 = tf.compat.v1.placeholder(tf.float32, shape=[None, 20, 7, 7], name="i9")
# i10 = tf.compat.v1.placeholder(tf.float32, shape=[None, 20, 7, 7], name="i10")

tf_buffer = [
    np.zeros([1, 3, 224, 224]),
    np.zeros([1, 3, 56, 56]),
    np.zeros([1, 4, 28, 28]),
    np.zeros([1, 4, 28, 28]),
    np.zeros([1, 8, 14, 14]),
    np.zeros([1, 8, 14, 14]),
    np.zeros([1, 8, 14, 14]),
    np.zeros([1, 12, 14, 14]),
    np.zeros([1, 12, 14, 14]),
    np.zeros([1, 20, 7, 7]),
    np.zeros([1, 20, 7, 7])
]

# We can verify that we can access the list of operations in the graph
# for op in graph.get_operations():
#     if ':' in op.name:
#         print(op.name)
#     if '/i' in op.name:
#         print(op.name)
#     if '/o' in op.name:
#         print(op.name)
#     # prefix/Placeholder/inputs_placeholder
#     # ...
#     # prefix/Accuracy/predictions

# We access the input and output nodes
tf_inputs = []
tf_outputs = []
for i in range(len(tf_buffer)):
    tf_inputs.append(graph.get_tensor_by_name(f'prefix/i{i}:0'))
    tf_outputs.append(graph.get_tensor_by_name(f'prefix/o{i}:0'))

In [30]:
with tf.compat.v1.Session(graph=graph) as sess:
    # Note: we don't nee to initialize/restore anything
    # There is no Variables in this graph, only hardcoded constants 
    output = sess.run(tf_outputs, feed_dict={
        tf_inputs[i]: tf_buffer[i] for i in range(len(tf_buffer))
    })
    print(len(output))

11


In [31]:
with tf.compat.v1.Session(graph=graph) as sess:
    for _ in range(100):
        begin = time.time()
        output = sess.run(tf_outputs, feed_dict={
            tf_inputs[i]: tf_buffer[i] for i in range(len(tf_buffer))
        })
        print('Curr time (s):', time.time() - begin)

Curr time (s): 12.497068643569946
Curr time (s): 0.09468603134155273
Curr time (s): 0.09209609031677246
Curr time (s): 0.09480977058410645
Curr time (s): 0.11060786247253418
Curr time (s): 0.10247516632080078
Curr time (s): 0.10439920425415039
Curr time (s): 0.09397411346435547
Curr time (s): 0.09719991683959961
Curr time (s): 0.09632325172424316
Curr time (s): 0.09855294227600098
Curr time (s): 0.10595822334289551
Curr time (s): 0.09719967842102051
Curr time (s): 0.09566187858581543
Curr time (s): 0.10587716102600098
Curr time (s): 0.10362696647644043
Curr time (s): 0.09365177154541016
Curr time (s): 0.10292601585388184
Curr time (s): 0.09540820121765137
Curr time (s): 0.12858295440673828
Curr time (s): 0.11786913871765137
Curr time (s): 0.11094021797180176
Curr time (s): 0.11780095100402832
Curr time (s): 0.11663317680358887
Curr time (s): 0.11376500129699707
Curr time (s): 0.10424923896789551
Curr time (s): 0.11965608596801758


KeyboardInterrupt: 

### MobileNetV2 sanity check:

In [32]:
import torchvision

In [33]:
mobilenetv2_torch = torchvision.models.mobilenet_v2(pretrained=False, num_classes=27)

In [34]:
mobilenetv2_torch

MobileNetV2(
  (features): Sequential(
    (0): ConvBNReLU(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): ConvBNReLU(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): ConvBNReLU(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=Tr

In [35]:
torch_module

MobileNetV2(
  (features): ModuleList(
    (0): Sequential(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
        (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
        (3): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (4): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (1): BatchNorm2d(96, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (2): ReLU6(inplace=True)
       

In [36]:
MV2_MODEL_PATH = './models/mobilenet_v2.onnx'

torch_inputs = (torch.rand(1, 3, 224, 224, dtype=torch.float32))

for param in mobilenetv2_torch.parameters():
    param = param.float()

for module in mobilenetv2_torch.children():
    for module_1 in module.children():
        for module_2 in module_1.children():
            for module_3 in module_2.children():
                if hasattr(module_3, 'num_batches_tracked'):
                    module_3.num_batches_tracked = module_3.num_batches_tracked.float()
#                 if str(module_3).split('(')[0] == 'BatchNorm2d':
#                     print(module_3)

torch2onnx(
    torch_module=mobilenetv2_torch, 
    torch_inputs=torch_inputs, 
    onnx_path=MV2_MODEL_PATH
)

In [37]:
with open(MV2_MODEL_PATH, 'rb') as onnx_model_file:
    onnx_model = onnx.load_model(onnx_model_file)
onnx.checker.check_model(onnx_model)

In [38]:
# print(onnx.helper.printable_graph(onnx_model.graph))

In [39]:
model_simp, check = simplify(onnx_model)
assert check, "Simplified ONNX model could not be validated"

In [40]:
print('Before', onnx_model.ByteSize())
print('After', model_simp.ByteSize())

Before 9203331
After 8977491


In [41]:
# print(onnx.helper.printable_graph(model_simp.graph))

In [42]:
MV2_SIMPLE_MODEL_PATH = './models/mv2_simple.onnx'
onnx.save(model_simp, MV2_SIMPLE_MODEL_PATH)

In [43]:
import onnxruntime
ort_session = onnxruntime.InferenceSession(MV2_SIMPLE_MODEL_PATH)
ort_session

In [44]:
tf_rep = prepare(model_simp)
print(tf_rep.inputs) # Input nodes to the model
print('-----')
print(tf_rep.outputs) # Output nodes from the model
print('-----')
print(tf_rep.tensor_dict) # All nodes in the model

2020-06-28 23:38:30,849 - onnx-tf - INFO - Fail to get since_version of BitShift in domain `` with max_inclusive_version=10. Set to 1.
2020-06-28 23:38:30,850 - onnx-tf - INFO - Unknown op ConstantFill in domain `ai.onnx`.
2020-06-28 23:38:30,851 - onnx-tf - INFO - Fail to get since_version of CumSum in domain `` with max_inclusive_version=10. Set to 1.
2020-06-28 23:38:30,852 - onnx-tf - INFO - Fail to get since_version of Det in domain `` with max_inclusive_version=10. Set to 1.
2020-06-28 23:38:30,853 - onnx-tf - INFO - Fail to get since_version of DynamicQuantizeLinear in domain `` with max_inclusive_version=10. Set to 1.
2020-06-28 23:38:30,854 - onnx-tf - INFO - Fail to get since_version of GatherND in domain `` with max_inclusive_version=10. Set to 1.
2020-06-28 23:38:30,855 - onnx-tf - INFO - Unknown op ImageScaler in domain `ai.onnx`.
2020-06-28 23:38:30,857 - onnx-tf - INFO - Fail to get since_version of Range in domain `` with max_inclusive_version=10. Set to 1.
2020-06-28 2

['i0']
-----
['o0']
-----
{'classifier.1.bias': <tf.Tensor 'classifier.1.bias:0' shape=(27,) dtype=float32>, 'classifier.1.weight': <tf.Tensor 'classifier.1.weight:0' shape=(27, 1280) dtype=float32>, '467': <tf.Tensor '467:0' shape=(32, 3, 3, 3) dtype=float32>, '469': <tf.Tensor '469:0' shape=(32,) dtype=float32>, '471': <tf.Tensor '471:0' shape=(32, 1, 3, 3) dtype=float32>, '473': <tf.Tensor '473:0' shape=(32,) dtype=float32>, '475': <tf.Tensor '475:0' shape=(16, 32, 1, 1) dtype=float32>, '477': <tf.Tensor '477:0' shape=(16,) dtype=float32>, '479': <tf.Tensor '479:0' shape=(96, 16, 1, 1) dtype=float32>, '481': <tf.Tensor '481:0' shape=(96,) dtype=float32>, '483': <tf.Tensor '483:0' shape=(96, 1, 3, 3) dtype=float32>, '485': <tf.Tensor '485:0' shape=(96,) dtype=float32>, '487': <tf.Tensor '487:0' shape=(24, 96, 1, 1) dtype=float32>, '489': <tf.Tensor '489:0' shape=(24,) dtype=float32>, '491': <tf.Tensor '491:0' shape=(144, 24, 1, 1) dtype=float32>, '493': <tf.Tensor '493:0' shape=(144,

In [45]:
TF_SIMPLE_MODEL_PATH = './models/mv2_simple_tf.pb'
tf_rep.export_graph(TF_SIMPLE_MODEL_PATH)

* Keras -> tf.js MobileNetV2 sanity check:

In [4]:
keras_mv2 = tf.keras.applications.MobileNetV2(
    input_shape=None,
    alpha=1.0,
    include_top=False,
    weights="imagenet",
    input_tensor=None,
    pooling=None,
    classes=27,
    classifier_activation="softmax"
)

In [5]:
keras_mv2.compile()
keras_mv2.summary()

Model: "mobilenetv2_1.00_224"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 3, None, Non 0                                            
__________________________________________________________________________________________________
Conv1_pad (ZeroPadding2D)       (None, 3, None, None 0           input_1[0][0]                    
__________________________________________________________________________________________________
Conv1 (Conv2D)                  (None, 32, None, Non 864         Conv1_pad[0][0]                  
__________________________________________________________________________________________________
bn_Conv1 (BatchNormalization)   (None, 32, None, Non 128         Conv1[0][0]                      
_______________________________________________________________________________

In [6]:
keras_mv2.save('./models/keras_mv2.h5')  # creates a HDF5 file 'my_model.h5'
del keras_mv2  # deletes the existing model

# returns a compiled model
# identical to the previous one
keras_mv2 = load_model('./models/keras_mv2.h5')
keras_mv2.summary()

Model: "mobilenetv2_1.00_224"
__________________________________________________________________________________________________
Layer (type)                    Output Shape         Param #     Connected to                     
input_1 (InputLayer)            [(None, 3, None, Non 0                                            
__________________________________________________________________________________________________
Conv1_pad (ZeroPadding2D)       (None, 3, None, None 0           input_1[0][0]                    
__________________________________________________________________________________________________
Conv1 (Conv2D)                  (None, 32, None, Non 864         Conv1_pad[0][0]                  
__________________________________________________________________________________________________
bn_Conv1 (BatchNormalization)   (None, 32, None, Non 128         Conv1[0][0]                      
_______________________________________________________________________________

In [7]:
tfjs.converters.save_keras_model(keras_mv2, './models/keras_mv2/')

/Users/izakharkin/opt/anaconda3/envs/tf20/lib/python3.7/site-packages/tensorflowjs/converters/keras_h5_conversion.py:122: H5pyDeprecationWarning: The default file mode will change to 'r' (read-only) in h5py 3.0. To suppress this warning, pass the mode you need to h5py.File(), or set the global default h5.get_config().default_file_mode, or set the environment variable H5PY_DEFAULT_READONLY=1. Available modes are: 'r', 'r+', 'w', 'w-'/'x', 'a'. See the docs for details.
  return h5py.File(h5file)


In [26]:
* проверь mmdnn для перегона в onnx и для перегона pytorch2keras
* сравни mobilenetv2 и tsm построчно

SyntaxError: invalid syntax (<ipython-input-26-9e7c6ff808a1>, line 1)

Can help: 
* [tf.js webcam demo](https://github.com/tensorflow/tfjs-examples/tree/master/webcam-transfer-learning)
* [tf.js native mobilenet](https://github.com/tensorflow/tfjs-models/tree/master/mobilenet)
* [tf.js-converter mobilenet demo](https://github.com/tensorflow/tfjs/tree/master/tfjs-converter/demo/mobilenet) (converted from TF)
* https://pytorch.org/docs/master/onnx.html#tracing-vs-scripting

---

### Run model in camera loop:

* PyTorch loop:

In [46]:
# WINDOW_NAME = 'Video Gesture Recognition'

# print("Open camera...")
# cap = cv2.VideoCapture(0)

# print(cap)

# # set a lower resolution for speed up
# cap.set(cv2.CAP_PROP_FRAME_WIDTH, 320)
# cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 240)

# # env variables
# full_screen = False
# cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)
# cv2.resizeWindow(WINDOW_NAME, 640, 480)
# cv2.moveWindow(WINDOW_NAME, 0, 0)
# cv2.setWindowTitle(WINDOW_NAME, WINDOW_NAME)


# t = None
# index = 0
# buffer = (
#     torch.zeros([1, 3, 56, 56]),
#     torch.zeros([1, 4, 28, 28]),
#     torch.zeros([1, 4, 28, 28]),
#     torch.zeros([1, 8, 14, 14]),
#     torch.zeros([1, 8, 14, 14]),
#     torch.zeros([1, 8, 14, 14]),
#     torch.zeros([1, 12, 14, 14]),
#     torch.zeros([1, 12, 14, 14]),
#     torch.zeros([1, 20, 7, 7]),
#     torch.zeros([1, 20, 7, 7])
# )

# idx = 0
# history = [2, 2]
# history_logit = []
# history_timing = []

# i_frame = -1

# print("Ready!")
# while True:
#     i_frame += 1
#     _, img = cap.read()  # (480, 640, 3) 0 ~ 255
#     if i_frame % 2 == 0:  # skip every other frame to obtain a suitable frame rate
#         t1 = time.time()
#         img_tran = torch.from_numpy(transform(img)).float().contiguous()
#         with torch.no_grad():
#             outputs = torch_module(img_tran, *buffer)  # was input_var earlier
#             feat, buffer = outputs[0], outputs[1:]
#         if SOFTMAX_THRES > 0:
#             feat_np = feat.numpy().reshape(-1)
#             feat_np -= feat_np.max()
#             softmax = np.exp(feat_np) / np.sum(np.exp(feat_np))
#             print(max(softmax))
#             if max(softmax) > SOFTMAX_THRES:
#                 idx_ = np.argmax(feat.numpy(), axis=1)[0]
#             else:
#                 idx_ = idx
#         else:
#             idx_ = np.argmax(feat.numpy(), axis=1)[0]
#         if HISTORY_LOGIT:
#             history_logit.append(feat.numpy())
#             history_logit = history_logit[-12:]
#             avg_logit = sum(history_logit)
#             idx_ = np.argmax(avg_logit, axis=1)[0]
#         idx, history = process_output(idx_, history)
#         t2 = time.time()
#         print(f"{index} {catigories[idx]}")
#         current_time = t2 - t1
#     img = cv2.resize(img, (640, 480))
#     img = img[:, ::-1]
#     height, width, _ = img.shape
#     label = np.zeros([height // 10, width, 3]).astype('uint8') + 255
#     cv2.putText(label, 'Prediction: ' + catigories[idx],
#                 (0, int(height / 16)),
#                 cv2.FONT_HERSHEY_SIMPLEX,
#                 0.7, (0, 0, 0), 2)
#     cv2.putText(label, '{:.1f} Vid/s'.format(1 / current_time),
#                 (width - 170, int(height / 16)),
#                 cv2.FONT_HERSHEY_SIMPLEX,
#                 0.7, (0, 0, 0), 2)
#     img = np.concatenate((img, label), axis=0)
#     cv2.imshow(WINDOW_NAME, img)
#     key = cv2.waitKey(1)
#     if key & 0xFF == ord('q') or key == 27:  # exit
#         break
#     elif key == ord('F') or key == ord('f'):  # full screen
#         print('Changing full screen option!')
#         full_screen = not full_screen
#         if full_screen:
#             print('Setting FS!!!')
#             cv2.setWindowProperty(WINDOW_NAME, cv2.WND_PROP_FULLSCREEN,
#                                   cv2.WINDOW_FULLSCREEN)
#         else:
#             cv2.setWindowProperty(WINDOW_NAME, cv2.WND_PROP_FULLSCREEN,
#                                   cv2.WINDOW_NORMAL)
#     if t is None:
#         t = time.time()
#     else:
#         nt = time.time()
#         index += 1
#         t = nt

In [47]:
# cap.release()
# cv2.destroyAllWindows()

* ONNX Runtime loop:

In [59]:
import onnxruntime
ort_session = onnxruntime.InferenceSession(ONNX_SIMPLE_MODEL_PATH)
input_names = [ort_session.get_inputs()[i].name for i in range(len(ort_session.get_inputs()))]
output_names = [ort_session.get_outputs()[i].name for i in range(len(ort_session.get_outputs()))]

WINDOW_NAME = 'Video Gesture Recognition'

print("Open camera...")
cap = cv2.VideoCapture(0)

print(cap)

# set a lower resolution for speed up
cap.set(cv2.CAP_PROP_FRAME_WIDTH, 320)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, 240)

# env variables
full_screen = False
cv2.namedWindow(WINDOW_NAME, cv2.WINDOW_NORMAL)
cv2.resizeWindow(WINDOW_NAME, 640, 480)
cv2.moveWindow(WINDOW_NAME, 0, 0)
cv2.setWindowTitle(WINDOW_NAME, WINDOW_NAME)


t = None
index = 0
np_buffer = [
    np.zeros([1, 3, 56, 56]),
    np.zeros([1, 4, 28, 28]),
    np.zeros([1, 4, 28, 28]),
    np.zeros([1, 8, 14, 14]),
    np.zeros([1, 8, 14, 14]),
    np.zeros([1, 8, 14, 14]),
    np.zeros([1, 12, 14, 14]),
    np.zeros([1, 12, 14, 14]),
    np.zeros([1, 20, 7, 7]),
    np.zeros([1, 20, 7, 7])
]
np_buffer = {f'i{i+1}': x.astype(np.float32) for i, x in enumerate(np_buffer)}

idx = 0
history = [2, 2]
history_logit = []
history_timing = []

i_frame = -1

print("Ready!")
while True:
    i_frame += 1
    _, img = cap.read()  # (480, 640, 3) 0 ~ 255
    if i_frame % 2 == 0:  # skip every other frame to obtain a suitable frame rate
        t1 = time.time()
        img_tran = transform(img)
        with torch.no_grad():
            np_buffer.update({'i0': img_tran})
            np_buffer = {k: v.astype(np.float32) for k, v in np_buffer.items()}
            outputs = ort_session.run(output_names, np_buffer)
            feat, np_buffer_values = outputs[0], outputs[1:]
            np_buffer = {f'i{i+1}': np_buffer_values[i] for i in range(len(np_buffer_values))}
        if SOFTMAX_THRES > 0:
            feat_np = feat.reshape(-1)
            feat_np -= feat_np.max()
            softmax = np.exp(feat_np) / np.sum(np.exp(feat_np))
            print(max(softmax))
            if max(softmax) > SOFTMAX_THRES:
                idx_ = np.argmax(feat, axis=1)[0]
            else:
                idx_ = idx
        else:
            idx_ = np.argmax(feat, axis=1)[0]
        if HISTORY_LOGIT:
            history_logit.append(feat)
            history_logit = history_logit[-12:]
            avg_logit = sum(history_logit)
            idx_ = np.argmax(avg_logit, axis=1)[0]
        idx, history = process_output(idx_, history)
        t2 = time.time()
        print(f"{index} {catigories[idx]}")
        current_time = t2 - t1
    img = cv2.resize(img, (640, 480))
    img = img[:, ::-1]
    height, width, _ = img.shape
    label = np.zeros([height // 10, width, 3]).astype('uint8') + 255
    cv2.putText(label, 'Prediction: ' + catigories[idx],
                (0, int(height / 16)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7, (0, 0, 0), 2)
    cv2.putText(label, '{:.1f} Vid/s'.format(1 / current_time),
                (width - 170, int(height / 16)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.7, (0, 0, 0), 2)
    img = np.concatenate((img, label), axis=0)
    cv2.imshow(WINDOW_NAME, img)
    key = cv2.waitKey(1)
    if key & 0xFF == ord('q') or key == 27:  # exit
        break
    elif key == ord('F') or key == ord('f'):  # full screen
        print('Changing full screen option!')
        full_screen = not full_screen
        if full_screen:
            print('Setting FS!!!')
            cv2.setWindowProperty(WINDOW_NAME, cv2.WND_PROP_FULLSCREEN,
                                  cv2.WINDOW_FULLSCREEN)
        else:
            cv2.setWindowProperty(WINDOW_NAME, cv2.WND_PROP_FULLSCREEN,
                                  cv2.WINDOW_NORMAL)
    if t is None:
        t = time.time()
    else:
        nt = time.time()
        index += 1
        t = nt

Open camera...
<VideoCapture 0x16c796990>
Ready!
0 No gesture
1 Shaking Hand
3 Shaking Hand
5 Shaking Hand
7 Shaking Hand
9 Shaking Hand
11 Shaking Hand
13 Shaking Hand
15 Shaking Hand
17 Shaking Hand
19 Shaking Hand
21 Shaking Hand
23 Shaking Hand
25 Shaking Hand
27 Shaking Hand
29 Shaking Hand
31 Shaking Hand
33 Swiping Left
35 Swiping Left
37 Swiping Left
39 Swiping Left
41 Swiping Left
43 Swiping Left
45 Swiping Left
47 Swiping Left
49 Swiping Left
51 Swiping Left
53 Swiping Left
55 Swiping Right
57 Swiping Right
59 Swiping Right
61 Swiping Right
63 Swiping Right
65 Swiping Right
67 Swiping Right
69 Swiping Left
71 Swiping Left
73 Swiping Left
75 Swiping Left
77 Swiping Left
79 Swiping Left
81 Swiping Left
83 Swiping Left
85 Swiping Left
87 Swiping Left
89 Swiping Left
91 Swiping Right
93 Swiping Right
95 Swiping Right
97 Swiping Right
99 Swiping Right
101 Swiping Right
103 Swiping Right
105 Stop Sign
107 Stop Sign
109 Stop Sign
111 Zooming Out With Full Hand
113 Zooming Out With F

843 No gesture
845 No gesture
847 No gesture
849 No gesture
851 No gesture
853 No gesture
855 No gesture
857 No gesture
859 No gesture
861 No gesture
863 No gesture
865 No gesture
867 No gesture
869 No gesture
871 No gesture
873 No gesture
875 No gesture
877 No gesture
879 No gesture
881 No gesture
883 No gesture
885 No gesture
887 No gesture
889 No gesture
891 No gesture
893 No gesture
895 No gesture
897 No gesture
899 No gesture
901 No gesture
903 No gesture
905 No gesture
907 No gesture
909 No gesture
911 No gesture
913 No gesture
915 No gesture
917 No gesture
919 No gesture
921 No gesture
923 No gesture
925 No gesture
927 No gesture
929 No gesture
931 No gesture
933 No gesture
935 No gesture
937 No gesture
939 No gesture
941 No gesture
943 No gesture
945 No gesture
947 No gesture
949 No gesture
951 No gesture
953 No gesture
955 No gesture
957 No gesture
959 No gesture
961 No gesture
963 No gesture
965 No gesture
967 No gesture
969 No gesture
971 No gesture
973 No gesture
975 No ges

1635 Swiping Down
1637 Swiping Down
1639 Swiping Down
1641 Swiping Down
1643 Swiping Down
1645 Swiping Down
1647 Swiping Down
1649 Swiping Down
1651 Swiping Down
1653 Swiping Down
1655 Swiping Down
1657 Swiping Down
1659 Swiping Down
1661 Swiping Down
1663 Shaking Hand
1665 Shaking Hand
1667 Shaking Hand
1669 Shaking Hand
1671 Shaking Hand
1673 Shaking Hand
1675 Shaking Hand
1677 Shaking Hand
1679 Shaking Hand
1681 Shaking Hand
1683 Shaking Hand
1685 Shaking Hand
1687 Shaking Hand
1689 Drumming Fingers
1691 Drumming Fingers
1693 Drumming Fingers
1695 Drumming Fingers
1697 Drumming Fingers
1699 Drumming Fingers
1701 Drumming Fingers
1703 Drumming Fingers
1705 Drumming Fingers
1707 Drumming Fingers
1709 Drumming Fingers
1711 Swiping Left
1713 Swiping Left
1715 Swiping Left
1717 Swiping Left
1719 Swiping Left
1721 Swiping Left
1723 Swiping Left
1725 Swiping Right
1727 Swiping Right
1729 Swiping Right
1731 Swiping Right
1733 Swiping Right
1735 Swiping Right
1737 Swiping Right
1739 Swiping 

2243 Sliding Two Fingers Left
2245 Sliding Two Fingers Left
2247 Sliding Two Fingers Right
2249 Sliding Two Fingers Right
2251 Sliding Two Fingers Right
2253 Sliding Two Fingers Right
2255 Sliding Two Fingers Right
2257 Sliding Two Fingers Right
2259 No gesture
2261 No gesture
2263 Drumming Fingers
2265 Drumming Fingers
2267 Drumming Fingers
2269 Drumming Fingers
2271 Drumming Fingers
2273 Drumming Fingers
2275 Drumming Fingers
2277 Drumming Fingers
2279 Drumming Fingers
2281 Swiping Left
2283 Swiping Left
2285 Swiping Left
2287 Swiping Left
2289 Swiping Left
2291 Swiping Left
2293 Swiping Left
2295 Swiping Left
2297 Swiping Right
2299 Swiping Right
2301 Swiping Right
2303 Pushing Hand Away
2305 Pushing Hand Away
2307 Stop Sign
2309 Stop Sign
2311 Stop Sign
2313 Stop Sign
2315 Stop Sign
2317 Stop Sign
2319 Stop Sign
2321 Stop Sign
2323 Stop Sign
2325 Stop Sign
2327 Stop Sign
2329 Stop Sign
2331 Stop Sign
2333 Stop Sign
2335 Stop Sign
2337 Pushing Hand Away
2339 Pushing Hand Away
2341 P

2851 Zooming Out With Full Hand
2853 Zooming Out With Full Hand
2855 No gesture
2857 No gesture
2859 No gesture
2861 No gesture
2863 No gesture
2865 No gesture
2867 No gesture
2869 No gesture
2871 No gesture
2873 No gesture
2875 No gesture
2877 No gesture
2879 No gesture
2881 No gesture
2883 No gesture
2885 No gesture
2887 No gesture
2889 No gesture
2891 No gesture
2893 No gesture
2895 No gesture
2897 No gesture
2899 No gesture
2901 No gesture
2903 No gesture
2905 No gesture
2907 No gesture
2909 No gesture
2911 No gesture
2913 No gesture
2915 No gesture
2917 No gesture
2919 No gesture
2921 No gesture
2923 No gesture
2925 No gesture
2927 No gesture
2929 No gesture
2931 No gesture
2933 No gesture
2935 No gesture
2937 No gesture
2939 No gesture
2941 No gesture
2943 No gesture
2945 No gesture
2947 No gesture
2949 No gesture
2951 No gesture
2953 No gesture
2955 No gesture
2957 No gesture
2959 No gesture
2961 No gesture
2963 No gesture
2965 No gesture
2967 No gesture
2969 No gesture
2971 No 

KeyboardInterrupt: 

In [58]:
cap.release()
cv2.destroyAllWindows()